In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, transforms

In [3]:
import sys
print(sys.executable)

/home/iztihad/venvs/ml/bin/python


In [4]:
model_config = {
    "batch_size": 16,
    "input_size": 224,
    "architecture": "resnet-50",
    "learning_rate": 0.001,
    "epochs": 10,
    "pretrained":True
}

In [5]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

train_dir = "../flowers/train"
val_dir = "../flowers/val"

train_dataset = datasets.ImageFolder(root=train_dir, transform=data_transforms["train"])
val_dataset = datasets.ImageFolder(root=val_dir, transform=data_transforms["val"])

train_dataloader = DataLoader(train_dataset, batch_size=model_config["batch_size"], shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=model_config["batch_size"], shuffle=False)

In [6]:
from torchvision.models import ResNet50_Weights

resnet50 = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

for param in resnet50.parameters():
    param.requires_grad = False

resnet50.fc = nn.Linear(resnet50.fc.in_features, 84)

for param in resnet50.fc.parameters():
    param.requires_grad = True

gpu = torch.device("cuda")
resnet50 = resnet50.to(gpu)


In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(resnet50.parameters(), lr=model_config["learning_rate"])
epochs = model_config["epochs"]

In [8]:
def train_model(model, train_dataloader, optimizer, criterion, epochs):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, label in train_dataloader:
            images = images.to(gpu)
            label = label.to(gpu)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, label)
            loss.backward()
            optimizer.step()

            total_loss = total_loss + loss.item()

        print(f"Epoch: {epoch + 1}, Training Loss: {total_loss/len(train_dataloader)}")

In [9]:
train_model(resnet50, train_dataloader, optimizer, criterion, model_config["epochs"])

Epoch: 1, Training Loss: 0.9362545230711599
Epoch: 2, Training Loss: 0.4350429820824429
Epoch: 3, Training Loss: 0.3508308032950557
Epoch: 4, Training Loss: 0.3038240455654988
Epoch: 5, Training Loss: 0.2662900333919848
Epoch: 6, Training Loss: 0.2396569431303032
Epoch: 7, Training Loss: 0.22053565487146853
Epoch: 8, Training Loss: 0.21177408149636598
Epoch: 9, Training Loss: 0.18236437414687467
Epoch: 10, Training Loss: 0.1799888928008982


In [10]:
def validate_model(model, val_dataloader):
    with torch.no_grad():
        model.eval()
        total = 0
        total_correct = 0

        for images, labels in val_dataloader:
            images = images.to(gpu)
            labels = labels.to(gpu)

            output = model(images)
            _, predicted = torch.max(output, 1)

            total = total + len(labels)
            total_correct = total_correct + (predicted == labels).sum().item()

        return total_correct/total 



In [11]:
accuracy = validate_model(resnet50, val_dataloader)
print(f"Accuracy: {100 * accuracy}")

Accuracy: 93.6696340257171
